# Домашняя работа 3
# Динамическое программирование

Требуется выполнить три задания:

1. В алгоритме Policy Iteration важным гиперпараметром является gamma. Требуется ответить на вопрос, какой gamma лучше выбирать. Качество обученной политики можно оценивать например запуская среду 1000 раз и взяв после этого средний total_reward.

2. На шаге Policy Evaluation мы каждый раз начинаем с нулевых values. А что будет если вместо этого начинать с values обученных на предыдущем шаге? Будет ли алгоритм работать? Если да, то будет ли он работать лучше?

3. Написать Value Iteriation. Исследовать гиперпараметры (в том числе gamma). Cравнить с Policy Iteration. Поскольку в Policy Iteration есть еще внутренний цикл, то адекватным сравнением алгоритмов будет не графики их результативности относительно внешнего цикла, а графики относительно, например, количества обращения к среде.

## Задание 1. Policy Iteration, выбор параметра gamma

In [1]:
import numpy as np
import time
from Frozen_Lake import FrozenLakeEnv

env = FrozenLakeEnv()

Если награда не зависит от $s'$:
$$
q(s,a) = R(s, a) + \gamma \sum_{s'} P(s'|s,a) v(s')
$$
Если награда зависит от $s'$:
$$
q(s,a) = \sum_{s'} P(s'|s,a) \Big( R(s,a,s') + \gamma v(s')\Big)
$$
Если награда не зависит от $s'$ - это частный случай того, когда награда зависит от $s'$:
$$
q(s,a) = \sum_{s'} P(s'|s,a) \Big( R(s,a) + \gamma v(s')\Big) =$$
$$= R(s,a) \sum_{s'} P(s'|s,a) + \gamma \sum_{s'} P(s'|s,a) v(s') =$$
$$ = R(s, a) + \gamma \sum_{s'} P(s'|s,a) v(s')
$$



In [2]:
def get_q_values(v_values, gamma):
    q_values = {}
    for state in env.get_all_states():
        q_values[state] = {}
        for action in env.get_possible_actions(state):
            q_values[state][action] = 0
            for next_state in env.get_next_states(state, action):
                prob = env.get_transition_prob(state, action, next_state)
                reward = env.get_reward(state, action, next_state)
                q_values[state][action] += prob * (reward + gamma * v_values[next_state])

    return q_values

In [3]:
def init_policy():
    policy = {}
    for state in env.get_all_states():
        policy[state] = {}
        for action in env.get_possible_actions(state):
            policy[state][action] = 1 / len(env.get_possible_actions(state))
    return policy

In [4]:
def init_v_values():
    v_values = {}
    for state in env.get_all_states():
        v_values[state] = 0
    return v_values

In [5]:
def policy_evaluation_step(v_values, policy, gamma):
    q_values = get_q_values(v_values, gamma)
    new_v_values = init_v_values()
    for state in env.get_all_states():
        new_v_values[state] = 0
        for action in env.get_possible_actions(state):
            new_v_values[state] += policy[state][action] * q_values[state][action]
    return new_v_values

In [6]:
def policy_evaluation(policy, gamma, eval_iter_n, min_diff):
    v_values = init_v_values()
    for i in range(eval_iter_n):
        new_v_values = policy_evaluation_step(v_values, policy, gamma)

        diff = max(abs(new_v_values[state] - v_values[state])
                   for state in env.get_all_states())

        v_values = new_v_values

        if diff < min_diff:
            break

    q_values = get_q_values(v_values, gamma)
    return q_values

In [7]:
def policy_improvement(q_values):
    policy = {}
    for state in env.get_all_states():
        policy[state] = {}
        argmax_action = None
        max_q_value = float('-inf')
        for action in env.get_possible_actions(state):
            policy[state][action] = 0
            if q_values[state][action] > max_q_value:
                argmax_action = action
                max_q_value = q_values[state][action]
        policy[state][argmax_action] = 1
    return policy

In [8]:
def get_mean_reward(policy):
    total_rewards = []

    for _ in range(1000):
        total_reward = 0
        state = env.reset()
        for _ in range(1000):
            action = np.random.choice(env.get_possible_actions(state),
                                      p=list(policy[state].values()))
            state, reward, done, _ = env.step(action)
            total_reward += reward

            if done:
                break

        total_rewards.append(total_reward)

    return np.mean(total_rewards)

In [ ]:
# POLICY ITERATION

iter_n = 100
eval_iter_n = 100
min_diff = 0.001
gamma = 0.999

policy = init_policy()
for _ in range(iter_n):
    q_values = policy_evaluation(policy, gamma, eval_iter_n, min_diff)
    policy = policy_improvement(q_values)
mean_reward = get_mean_reward(policy)
print('mean_reward =', mean_reward)

mean_reward = 0.876


In [ ]:
policy

{(0, 0): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (0, 1): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (0, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (0, 3): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (1, 0): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (1, 1): {None: 1},
 (1, 2): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (1, 3): {None: 1},
 (2, 0): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (2, 1): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (2, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (2, 3): {None: 1},
 (3, 0): {None: 1},
 (3, 1): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 2): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 3): {None: 1}}

In [ ]:
state = env.reset()
total_reward = 0
for _ in range(1000):
    action = np.random.choice(env.get_possible_actions(state),
                              p=list(policy[state].values()))
    state, reward, done, _ = env.step(action)
    total_reward += reward

    env.render()
    time.sleep(0.5)

    if done:
        break
print('total_reward =', total_reward)

*FFF
FHFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
FHFH
*FFH
HFFG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
FFFH
H*FG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
FFFH
H*FG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
*FFH
HFFG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
FFFH
H*FG

SFFF
FHFH
FFFH
HF*G

SFFF
FHFH
FFFH
HFF*

total_reward = 1.0


In [ ]:
# POLICY ITERATION

iter_n = 100
eval_iter_n = 100
min_diff = 0.0001

gamma_param = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6,
               0.7, 0.8, 0.9, 0.99, 0.999, 1.0]
for gamma in gamma_param:
    policy = init_policy()
    for _ in range(iter_n):
        q_values = policy_evaluation(policy, gamma, eval_iter_n, min_diff)
        policy = policy_improvement(q_values)
    mean_reward = get_mean_reward(policy)
    print('gamma =', gamma, '\t mean_reward =', mean_reward)

gamma = 0.0 	 mean_reward = 0.0
gamma = 0.1 	 mean_reward = 0.736
gamma = 0.2 	 mean_reward = 0.733
gamma = 0.3 	 mean_reward = 0.732
gamma = 0.4 	 mean_reward = 0.74
gamma = 0.5 	 mean_reward = 0.747
gamma = 0.6 	 mean_reward = 0.763
gamma = 0.7 	 mean_reward = 0.714
gamma = 0.8 	 mean_reward = 0.702
gamma = 0.9 	 mean_reward = 0.731
gamma = 0.99 	 mean_reward = 0.852
gamma = 0.999 	 mean_reward = 0.876
gamma = 1.0 	 mean_reward = 0.991


Наибольшее значение награды получается при gamma = 1.0

## Задание 2. Policy Iteration, выбор значений v_values

In [ ]:
def policy_evaluation_new(v_values, policy, gamma, eval_iter_n, min_diff):
    for i in range(eval_iter_n):
        new_v_values = policy_evaluation_step(v_values, policy, gamma)

        diff = max(abs(new_v_values[state] - v_values[state])
                   for state in env.get_all_states())

        v_values = new_v_values

        if diff < min_diff:
            break

    q_values = get_q_values(new_v_values, gamma)
    return new_v_values, q_values

In [ ]:
# POLICY ITERATION

iter_n = 100
eval_iter_n = 100
min_diff = 0.001
gamma = 0.999

policy = init_policy()
v_values = init_v_values()

for _ in range(iter_n):
    v_values, q_values = policy_evaluation_new(v_values,
                                policy, gamma, eval_iter_n, min_diff)
    policy = policy_improvement(q_values)
mean_reward = get_mean_reward(policy)
print('mean_reward =', mean_reward)

mean_reward = 0.967


In [ ]:
policy

{(0, 0): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (0, 1): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (0, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (0, 3): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (1, 0): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (1, 1): {None: 1},
 (1, 2): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (1, 3): {None: 1},
 (2, 0): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (2, 1): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (2, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (2, 3): {None: 1},
 (3, 0): {None: 1},
 (3, 1): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 2): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 3): {None: 1}}

In [ ]:
state = env.reset()
total_reward = 0
for _ in range(1000):
    action = np.random.choice(env.get_possible_actions(state),
                              p=list(policy[state].values()))
    state, reward, done, _ = env.step(action)
    total_reward += reward

    env.render()
    time.sleep(0.5)

    if done:
        break
print('total_reward =', total_reward)

S*FF
FHFH
FFFH
HFFG

S*FF
FHFH
FFFH
HFFG

SF*F
FHFH
FFFH
HFFG

S*FF
FHFH
FFFH
HFFG

S*FF
FHFH
FFFH
HFFG

SF*F
FHFH
FFFH
HFFG

S*FF
FHFH
FFFH
HFFG

SF*F
FHFH
FFFH
HFFG

SFFF
FH*H
FFFH
HFFG

SFFF
FHFH
FF*H
HFFG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
FFFH
H*FG

SFFF
FHFH
FFFH
HF*G

SFFF
FHFH
FFFH
HFF*

total_reward = 1.0


Если использовать значения, полученные на предыдущем шаге, то алгоритм работает с небольшой модификацией и результат получается точнее.

## Задание 3. Value Iteration


$$Q_{i}(s, a) = \sum_{s'} P(s' | s,a) \cdot [ R(s,a,s') + \gamma V_{i}(s')]$$


 $$V_{(i+1)}(s) = \max_a \sum_{s'} P(s' | s,a) \cdot [ R(s,a,s') + \gamma V_{i}(s')] = \max_a Q_i(s,a)$$


 $$\pi^*(s) = arg \max_a \sum_{s'} P(s' | s,a) \cdot [ R(s,a,s') + \gamma V^*(s')] = arg \max_a Q^*(s,a)$$

In [20]:
def value_iteration(gamma, iter_n, min_diff):
    v_values = init_v_values()
    for iter in range(iter_n):
        q_values = get_q_values(v_values, gamma)
        new_v_values = {}
        for state in env.get_all_states():
            if env.is_terminal(state):
                new_v_values[state] = 0
            else:
                new_v_values[state] = max(q_values[state].values())

        diff = max(abs(new_v_values[state] - v_values[state])
                   for state in env.get_all_states())
        print('iter =', iter, '\t diff =', diff)

        v_values = new_v_values
        if diff < min_diff:
            break

    opt_policy = policy_improvement(q_values)

    return opt_policy


In [23]:
# VALUE ITERATION

iter_n = 100
min_diff = 0.001
gamma = 0.999

policy = init_policy()
policy = value_iteration(gamma, iter_n, min_diff)
mean_reward = get_mean_reward(policy)
print('mean_reward =', mean_reward)

iter = 0 	 diff = 0.8
iter = 1 	 diff = 0.63936
iter = 2 	 diff = 0.574848576
iter = 3 	 diff = 0.45941898193920006
iter = 4 	 diff = 0.36716765036580873
iter = 5 	 diff = 0.32604487352483813
iter = 6 	 diff = 0.14168769046332114
iter = 7 	 diff = 0.13731589234516328
iter = 8 	 diff = 0.05467638995109958
iter = 9 	 diff = 0.04358046349528011
iter = 10 	 diff = 0.023812687421132073
iter = 11 	 diff = 0.017358705382088613
iter = 12 	 diff = 0.016769500798975878
iter = 13 	 diff = 0.014792119015592409
iter = 14 	 diff = 0.01333867959452606
iter = 15 	 diff = 0.011818809398682295
iter = 16 	 diff = 0.010579339506452956
iter = 17 	 diff = 0.009411972212139741
iter = 18 	 diff = 0.008425747458377097
iter = 19 	 diff = 0.007530665185182528
iter = 20 	 diff = 0.006759124258247651
iter = 21 	 diff = 0.006069146851944818
iter = 22 	 diff = 0.005466707238564905
iter = 23 	 diff = 0.00493028241378779
iter = 24 	 diff = 0.004457482443670502
iter = 25 	 diff = 0.00403610989032388
iter = 26 	 diff = 

In [26]:
policy

{(0, 0): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (0, 1): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (0, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (0, 3): {'left': 0, 'down': 0, 'right': 0, 'up': 1},
 (1, 0): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (1, 1): {None: 1},
 (1, 2): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (1, 3): {None: 1},
 (2, 0): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (2, 1): {'left': 0, 'down': 1, 'right': 0, 'up': 0},
 (2, 2): {'left': 1, 'down': 0, 'right': 0, 'up': 0},
 (2, 3): {None: 1},
 (3, 0): {None: 1},
 (3, 1): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 2): {'left': 0, 'down': 0, 'right': 1, 'up': 0},
 (3, 3): {None: 1}}

In [24]:
state = env.reset()
total_reward = 0
for _ in range(1000):
    action = np.random.choice(env.get_possible_actions(state),
                              p=list(policy[state].values()))
    state, reward, done, _ = env.step(action)
    total_reward += reward

    env.render()
    time.sleep(0.5)

    if done:
        break
print('total_reward =', total_reward)

SFFF
*HFH
FFFH
HFFG

*FFF
FHFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

*FFF
FHFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
*HFH
FFFH
HFFG

SFFF
FHFH
*FFH
HFFG

SFFF
FHFH
F*FH
HFFG

SFFF
FHFH
FFFH
H*FG

SFFF
FHFH
FFFH
HF*G

SFFF
FHFH
FFFH
HFF*

total_reward = 1.0


Алгоритм требует гораздо меньшего количества операций за счет отсутствия внутреннего цикла. Решение уравнения Беллмана для v_valus происводится сразу на оптимальных значениях q_values.
